# Model Comparison and Ensemble Methods

## Advanced Model Selection and Combination Strategies

This notebook focuses on:
- Cross-validation for time series
- Model comparison with statistical tests
- Dynamic ensemble weighting
- Stacking and meta-learning
- Model selection criteria
- Forecast combination methods

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from typing import Dict, Any, List, Tuple, Optional
import warnings

warnings.filterwarnings("ignore")

# Statistical tests
from scipy import stats
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.diagnostic import acorr_ljungbox

# Machine learning
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

# Optimization
from scipy.optimize import minimize

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# Set style
plt.style.use("seaborn-v0_8-darkgrid")

## 1. Time Series Cross-Validation

In [ ]:
class TimeSeriesCrossValidator:
    """Advanced cross-validation for time series models."""

    def __init__(self, n_splits: int = 5, test_size: int = 30, gap: int = 0):
        """Initialize cross-validator.

        Parameters:
        -----------
        n_splits : int
            Number of splits for cross-validation
        test_size : int
            Size of test set in each split
        gap : int
            Gap between training and test set
        """
        self.n_splits = n_splits
        self.test_size = test_size
        self.gap = gap

    def split(self, data: pd.Series) -> List[Tuple[pd.Series, pd.Series]]:
        """Generate train/test splits for time series.

        Returns:
        --------
        List of (train, test) tuples
        """
        splits = []
        n = len(data)

        # Calculate split positions
        test_starts = np.linspace(
            n - self.test_size * self.n_splits,
            n - self.test_size,
            self.n_splits,
            dtype=int,
        )

        for test_start in test_starts:
            train_end = test_start - self.gap
            test_end = test_start + self.test_size

            if train_end > 0 and test_end <= n:
                train = data.iloc[:train_end]
                test = data.iloc[test_start:test_end]
                splits.append((train, test))

        return splits

    def evaluate_model(
        self, model_func, data: pd.Series, params: Dict = None
    ) -> Dict[str, Any]:
        """Evaluate a model using cross-validation.

        Parameters:
        -----------
        model_func : callable
            Function that fits model and returns predictions
        data : pd.Series
            Time series data
        params : Dict
            Model parameters

        Returns:
        --------
        Dictionary with evaluation results
        """
        splits = self.split(data)
        results = []

        for i, (train, test) in enumerate(splits):
            try:
                # Fit model and get predictions
                predictions = model_func(train, len(test), params)

                # Calculate metrics
                mae = mean_absolute_error(test, predictions)
                rmse = np.sqrt(mean_squared_error(test, predictions))

                results.append(
                    {
                        "fold": i,
                        "mae": mae,
                        "rmse": rmse,
                        "predictions": predictions,
                        "actual": test,
                    }
                )

            except Exception as e:
                print(f"Error in fold {i}: {e}")
                continue

        # Aggregate results
        if results:
            return {
                "mean_mae": np.mean([r["mae"] for r in results]),
                "std_mae": np.std([r["mae"] for r in results]),
                "mean_rmse": np.mean([r["rmse"] for r in results]),
                "std_rmse": np.std([r["rmse"] for r in results]),
                "fold_results": results,
            }
        else:
            return None

## 2. Model Comparison with Statistical Tests

In [ ]:
class ModelComparator:
    """Statistical comparison of forecasting models."""

    def __init__(self, actual: pd.Series, forecasts: Dict[str, pd.Series]):
        """Initialize comparator.

        Parameters:
        -----------
        actual : pd.Series
            Actual values
        forecasts : Dict[str, pd.Series]
            Dictionary of model forecasts
        """
        self.actual = actual
        self.forecasts = forecasts

    def diebold_mariano_test(
        self, model1: str, model2: str, h: int = 1
    ) -> Dict[str, float]:
        """Diebold-Mariano test for forecast accuracy.

        Parameters:
        -----------
        model1, model2 : str
            Names of models to compare
        h : int
            Forecast horizon

        Returns:
        --------
        Test statistic and p-value
        """
        # Get forecasts
        f1 = self.forecasts[model1]
        f2 = self.forecasts[model2]

        # Align with actual
        common_index = self.actual.index.intersection(f1.index).intersection(f2.index)
        actual = self.actual.loc[common_index]
        f1 = f1.loc[common_index]
        f2 = f2.loc[common_index]

        # Calculate loss differential
        e1 = actual - f1
        e2 = actual - f2
        d = e1**2 - e2**2  # Squared error loss

        # Calculate test statistic
        mean_d = d.mean()
        var_d = d.var()
        n = len(d)

        # Adjust variance for autocorrelation
        if h > 1:
            for i in range(1, h):
                gamma_i = ((d[:-i] - mean_d) * (d[i:] - mean_d)).mean()
                var_d += 2 * gamma_i

        dm_stat = mean_d / np.sqrt(var_d / n)
        p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))

        return {
            "dm_statistic": dm_stat,
            "p_value": p_value,
            "conclusion": f"{model1} is {'better' if dm_stat < 0 else 'worse'} than {model2}"
            + f" (p={p_value:.4f})",
        }

    def friedman_test(self) -> Dict[str, Any]:
        """Friedman test for multiple model comparison.

        Returns:
        --------
        Test results
        """
        # Prepare data for Friedman test
        errors = []
        model_names = []

        for name, forecast in self.forecasts.items():
            common_index = self.actual.index.intersection(forecast.index)
            if len(common_index) > 0:
                actual = self.actual.loc[common_index]
                forecast_aligned = forecast.loc[common_index]
                abs_errors = np.abs(actual - forecast_aligned)
                errors.append(abs_errors.values)
                model_names.append(name)

        if len(errors) < 3:
            return {"error": "Need at least 3 models for Friedman test"}

        # Perform test
        statistic, p_value = friedmanchisquare(*errors)

        # Rank models
        mean_errors = [np.mean(e) for e in errors]
        rankings = np.argsort(mean_errors)

        return {
            "statistic": statistic,
            "p_value": p_value,
            "rankings": {model_names[i]: rank + 1 for rank, i in enumerate(rankings)},
            "significant_difference": p_value < 0.05,
        }

    def model_confidence_set(self, alpha: float = 0.05) -> List[str]:
        """Model Confidence Set (MCS) procedure.

        Parameters:
        -----------
        alpha : float
            Significance level

        Returns:
        --------
        List of models in the confidence set
        """
        models = list(self.forecasts.keys())
        confidence_set = models.copy()

        while len(confidence_set) > 1:
            # Calculate pairwise test statistics
            max_p_value = 0
            worst_model = None

            for model in confidence_set:
                p_values = []
                for other in confidence_set:
                    if model != other:
                        result = self.diebold_mariano_test(model, other)
                        p_values.append(result["p_value"])

                if p_values:
                    avg_p = np.mean(p_values)
                    if avg_p > max_p_value:
                        max_p_value = avg_p
                        worst_model = model

            # Remove worst model if significantly worse
            if max_p_value < alpha:
                confidence_set.remove(worst_model)
            else:
                break

        return confidence_set

## 3. Advanced Ensemble Methods

In [ ]:
class EnsembleForecaster:
    """Advanced ensemble methods for time series forecasting."""

    def __init__(self, forecasts: Dict[str, pd.Series], actual: pd.Series = None):
        """Initialize ensemble forecaster.

        Parameters:
        -----------
        forecasts : Dict[str, pd.Series]
            Dictionary of individual model forecasts
        actual : pd.Series, optional
            Actual values for weight optimization
        """
        self.forecasts = forecasts
        self.actual = actual
        self.weights = {}

    def simple_average(self) -> pd.Series:
        """Simple average ensemble."""
        # Get common index
        common_index = None
        for forecast in self.forecasts.values():
            if common_index is None:
                common_index = forecast.index
            else:
                common_index = common_index.intersection(forecast.index)

        # Calculate average
        ensemble = pd.Series(index=common_index, dtype=float)
        for date in common_index:
            values = [f.loc[date] for f in self.forecasts.values() if date in f.index]
            ensemble.loc[date] = np.mean(values)

        return ensemble

    def weighted_average(self, weights: Dict[str, float] = None) -> pd.Series:
        """Weighted average ensemble.

        Parameters:
        -----------
        weights : Dict[str, float]
            Model weights (must sum to 1)
        """
        if weights is None:
            weights = self.weights

        # Normalize weights
        total = sum(weights.values())
        weights = {k: v / total for k, v in weights.items()}

        # Get common index
        common_index = None
        for name, forecast in self.forecasts.items():
            if name in weights:
                if common_index is None:
                    common_index = forecast.index
                else:
                    common_index = common_index.intersection(forecast.index)

        # Calculate weighted average
        ensemble = pd.Series(index=common_index, dtype=float)
        for date in common_index:
            weighted_sum = 0
            for name, forecast in self.forecasts.items():
                if name in weights and date in forecast.index:
                    weighted_sum += weights[name] * forecast.loc[date]
            ensemble.loc[date] = weighted_sum

        return ensemble

    def optimize_weights(self, method: str = "mse") -> Dict[str, float]:
        """Optimize ensemble weights.

        Parameters:
        -----------
        method : str
            Optimization criterion ('mse', 'mae', 'mape')

        Returns:
        --------
        Optimal weights
        """
        if self.actual is None:
            raise ValueError("Actual values required for weight optimization")

        # Prepare data
        model_names = list(self.forecasts.keys())
        n_models = len(model_names)

        # Get common index
        common_index = self.actual.index
        for forecast in self.forecasts.values():
            common_index = common_index.intersection(forecast.index)

        # Create forecast matrix
        X = np.zeros((len(common_index), n_models))
        for i, name in enumerate(model_names):
            X[:, i] = self.forecasts[name].loc[common_index].values

        y = self.actual.loc[common_index].values

        # Objective function
        def objective(weights):
            predictions = X @ weights
            if method == "mse":
                return np.mean((y - predictions) ** 2)
            elif method == "mae":
                return np.mean(np.abs(y - predictions))
            elif method == "mape":
                return np.mean(np.abs((y - predictions) / y)) * 100

        # Constraints
        constraints = [
            {"type": "eq", "fun": lambda w: np.sum(w) - 1},  # Weights sum to 1
        ]
        bounds = [(0, 1) for _ in range(n_models)]  # Weights between 0 and 1

        # Initial guess
        x0 = np.ones(n_models) / n_models

        # Optimize
        result = minimize(
            objective, x0, method="SLSQP", bounds=bounds, constraints=constraints
        )

        # Store weights
        self.weights = {model_names[i]: result.x[i] for i in range(n_models)}

        return self.weights

    def stacking_ensemble(self, meta_model=None) -> pd.Series:
        """Stacking ensemble with meta-learner.

        Parameters:
        -----------
        meta_model : sklearn model, optional
            Meta-learner model

        Returns:
        --------
        Ensemble predictions
        """
        if self.actual is None:
            raise ValueError("Actual values required for stacking")

        if meta_model is None:
            meta_model = Ridge(alpha=0.1)

        # Prepare data
        model_names = list(self.forecasts.keys())

        # Get common index for training
        common_index = self.actual.index
        for forecast in self.forecasts.values():
            common_index = common_index.intersection(forecast.index)

        # Create feature matrix
        X = pd.DataFrame(index=common_index)
        for name in model_names:
            X[name] = self.forecasts[name].loc[common_index]

        y = self.actual.loc[common_index]

        # Fit meta-model
        meta_model.fit(X, y)

        # Make predictions
        ensemble = pd.Series(meta_model.predict(X), index=common_index)

        return ensemble

    def dynamic_ensemble(self, window_size: int = 30) -> pd.Series:
        """Dynamic ensemble with time-varying weights.

        Parameters:
        -----------
        window_size : int
            Size of rolling window for weight calculation

        Returns:
        --------
        Dynamic ensemble predictions
        """
        if self.actual is None:
            raise ValueError("Actual values required for dynamic ensemble")

        # Get common index
        common_index = self.actual.index
        for forecast in self.forecasts.values():
            common_index = common_index.intersection(forecast.index)

        ensemble = pd.Series(index=common_index, dtype=float)

        for i in range(window_size, len(common_index)):
            # Get window data
            window_index = common_index[i - window_size : i]

            # Calculate weights based on recent performance
            weights = {}
            for name, forecast in self.forecasts.items():
                window_errors = np.abs(
                    self.actual.loc[window_index] - forecast.loc[window_index]
                )
                # Inverse error weighting
                weights[name] = 1 / (window_errors.mean() + 1e-6)

            # Normalize weights
            total = sum(weights.values())
            weights = {k: v / total for k, v in weights.items()}

            # Calculate ensemble prediction
            current_date = common_index[i]
            weighted_sum = sum(
                weights[name] * forecast.loc[current_date]
                for name, forecast in self.forecasts.items()
            )
            ensemble.iloc[i] = weighted_sum

        return ensemble.dropna()

## 4. Generate Sample Data and Models

In [ ]:
# Generate sample time series
np.random.seed(42)
dates = pd.date_range(start="2020-01-01", end="2023-12-31", freq="D")

# Create complex time series
trend = np.linspace(100, 150, len(dates))
seasonal = 10 * np.sin(2 * np.pi * np.arange(len(dates)) / 365.25)
noise = np.random.normal(0, 5, len(dates))
ts = pd.Series(trend + seasonal + noise, index=dates, name="value")

# Create sample forecasts (simulate different models)
np.random.seed(42)
test_start = "2023-10-01"
test_data = ts[ts.index >= test_start]

# Simulate model forecasts with different characteristics
forecasts = {}

# Model 1: Good trend, poor seasonality
forecasts["model_1"] = test_data + np.random.normal(0, 3, len(test_data))

# Model 2: Good seasonality, poor trend
forecasts["model_2"] = test_data + np.random.normal(2, 2, len(test_data))

# Model 3: Balanced
forecasts["model_3"] = test_data + np.random.normal(1, 2.5, len(test_data))

# Model 4: High variance
forecasts["model_4"] = test_data + np.random.normal(0, 6, len(test_data))

# Model 5: Systematic bias
forecasts["model_5"] = test_data + 5 + np.random.normal(0, 2, len(test_data))

print(f"Created {len(forecasts)} model forecasts")
print(f"Test period: {test_data.index.min()} to {test_data.index.max()}")
print(f"Number of test observations: {len(test_data)}")

## 5. Cross-Validation Analysis

In [ ]:
# Demonstrate time series cross-validation
cv = TimeSeriesCrossValidator(n_splits=5, test_size=20, gap=5)

# Visualize CV splits
splits = cv.split(ts)

fig = go.Figure()

# Plot full series
fig.add_trace(
    go.Scatter(
        x=ts.index,
        y=ts.values,
        mode="lines",
        name="Full Series",
        line=dict(color="lightgray", width=1),
    )
)

# Plot CV splits
colors = px.colors.qualitative.Plotly
for i, (train, test) in enumerate(splits):
    fig.add_trace(
        go.Scatter(
            x=test.index,
            y=test.values,
            mode="lines",
            name=f"Test Split {i + 1}",
            line=dict(color=colors[i], width=2),
        )
    )

fig.update_layout(
    title="Time Series Cross-Validation Splits",
    xaxis_title="Date",
    yaxis_title="Value",
    height=400,
)
fig.show()

print(f"\nCross-validation configuration:")
print(f"  - Number of splits: {cv.n_splits}")
print(f"  - Test size per split: {cv.test_size}")
print(f"  - Gap between train/test: {cv.gap}")

## 6. Statistical Model Comparison

In [ ]:
# Initialize model comparator
comparator = ModelComparator(test_data, forecasts)

print("\n" + "=" * 60)
print("STATISTICAL MODEL COMPARISON")
print("=" * 60)

# Pairwise Diebold-Mariano tests
print("\n📊 Diebold-Mariano Tests:")
print("-" * 40)

model_pairs = [("model_1", "model_2"), ("model_1", "model_3"), ("model_2", "model_3")]

for model1, model2 in model_pairs:
    result = comparator.diebold_mariano_test(model1, model2)
    print(f"\n{model1} vs {model2}:")
    print(f"  DM statistic: {result['dm_statistic']:.4f}")
    print(f"  P-value: {result['p_value']:.4f}")
    print(f"  {result['conclusion']}")

In [ ]:
# Friedman test for all models
print("\n📊 Friedman Test (All Models):")
print("-" * 40)

friedman_result = comparator.friedman_test()
print(f"Test statistic: {friedman_result['statistic']:.4f}")
print(f"P-value: {friedman_result['p_value']:.4f}")
print(f"Significant difference: {friedman_result['significant_difference']}")
print("\nModel Rankings:")
for model, rank in sorted(friedman_result["rankings"].items(), key=lambda x: x[1]):
    print(f"  {rank}. {model}")

In [ ]:
# Model Confidence Set
print("\n📊 Model Confidence Set (α=0.05):")
print("-" * 40)

confidence_set = comparator.model_confidence_set(alpha=0.05)
print(f"Models in confidence set: {confidence_set}")
print(
    f"\nInterpretation: These models are not significantly different from the best model"
)

## 7. Ensemble Methods

In [ ]:
# Initialize ensemble forecaster
ensemble = EnsembleForecaster(forecasts, test_data)

# Create different ensemble forecasts
ensemble_forecasts = {}

# Simple average
ensemble_forecasts["simple_avg"] = ensemble.simple_average()

# Optimize weights
optimal_weights = ensemble.optimize_weights(method="mse")
ensemble_forecasts["optimal_weighted"] = ensemble.weighted_average(optimal_weights)

# Stacking ensemble
ensemble_forecasts["stacking"] = ensemble.stacking_ensemble()

# Dynamic ensemble
ensemble_forecasts["dynamic"] = ensemble.dynamic_ensemble(window_size=20)

print("\n" + "=" * 60)
print("ENSEMBLE METHODS")
print("=" * 60)

print("\n📊 Optimal Weights (MSE optimization):")
for model, weight in optimal_weights.items():
    print(f"  {model}: {weight:.4f}")

## 8. Compare All Methods

In [ ]:
# Combine individual and ensemble forecasts
all_forecasts = {**forecasts, **ensemble_forecasts}

# Calculate metrics for all methods
results = []

for name, forecast in all_forecasts.items():
    # Align with actual
    common_index = test_data.index.intersection(forecast.index)
    if len(common_index) == 0:
        continue

    actual_aligned = test_data.loc[common_index]
    forecast_aligned = forecast.loc[common_index]

    mae = mean_absolute_error(actual_aligned, forecast_aligned)
    rmse = np.sqrt(mean_squared_error(actual_aligned, forecast_aligned))

    results.append(
        {
            "Model": name,
            "Type": "Ensemble"
            if "avg" in name or "stacking" in name or "dynamic" in name
            else "Individual",
            "MAE": mae,
            "RMSE": rmse,
        }
    )

results_df = pd.DataFrame(results).sort_values("RMSE")

print("\n" + "=" * 60)
print("COMPREHENSIVE MODEL COMPARISON")
print("=" * 60)
print(results_df.to_string())

# Highlight winner
best = results_df.iloc[0]
print(f"\n🏆 Best Method: {best['Model']} ({best['Type']})")
print(f"   MAE: {best['MAE']:.4f}")
print(f"   RMSE: {best['RMSE']:.4f}")

## 9. Visualization of Results

In [ ]:
# Plot comparison of individual vs ensemble methods
fig = go.Figure()

# Plot actual values
fig.add_trace(
    go.Scatter(
        x=test_data.index,
        y=test_data.values,
        mode="lines",
        name="Actual",
        line=dict(color="black", width=2),
    )
)

# Plot best individual model
best_individual = results_df[results_df["Type"] == "Individual"].iloc[0]["Model"]
fig.add_trace(
    go.Scatter(
        x=forecasts[best_individual].index,
        y=forecasts[best_individual].values,
        mode="lines",
        name=f"Best Individual ({best_individual})",
        line=dict(color="blue", dash="dot"),
    )
)

# Plot ensemble methods
colors = ["red", "green", "orange", "purple"]
for i, (name, forecast) in enumerate(ensemble_forecasts.items()):
    fig.add_trace(
        go.Scatter(
            x=forecast.index,
            y=forecast.values,
            mode="lines",
            name=name.replace("_", " ").title(),
            line=dict(color=colors[i % len(colors)]),
        )
    )

fig.update_layout(
    title="Individual vs Ensemble Methods",
    xaxis_title="Date",
    yaxis_title="Value",
    hovermode="x unified",
    height=500,
)
fig.show()

In [ ]:
# Performance comparison visualization
fig = make_subplots(
    rows=1, cols=2, subplot_titles=("MAE Comparison", "RMSE Comparison")
)

# MAE comparison
fig.add_trace(
    go.Bar(
        x=results_df["Model"],
        y=results_df["MAE"],
        marker_color=["red" if t == "Ensemble" else "blue" for t in results_df["Type"]],
        name="MAE",
    ),
    row=1,
    col=1,
)

# RMSE comparison
fig.add_trace(
    go.Bar(
        x=results_df["Model"],
        y=results_df["RMSE"],
        marker_color=["red" if t == "Ensemble" else "blue" for t in results_df["Type"]],
        name="RMSE",
    ),
    row=1,
    col=2,
)

fig.update_xaxes(tickangle=45)
fig.update_layout(height=400, title="Model Performance Comparison", showlegend=False)
fig.show()

# Add legend
print("\n🔵 Blue bars: Individual models")
print("🔴 Red bars: Ensemble methods")

## 10. Key Insights and Recommendations

In [ ]:
# Generate insights
print("\n" + "=" * 60)
print("KEY INSIGHTS AND RECOMMENDATIONS")
print("=" * 60)

# Performance improvement from ensemble
best_individual_rmse = results_df[results_df["Type"] == "Individual"]["RMSE"].min()
best_ensemble_rmse = results_df[results_df["Type"] == "Ensemble"]["RMSE"].min()
improvement = (best_individual_rmse - best_ensemble_rmse) / best_individual_rmse * 100

print(f"\n📈 Performance Improvement:")
print(f"  Best individual RMSE: {best_individual_rmse:.4f}")
print(f"  Best ensemble RMSE: {best_ensemble_rmse:.4f}")
print(f"  Improvement: {improvement:.2f}%")

# Model diversity
print(f"\n🎯 Model Diversity:")
correlations = pd.DataFrame(forecasts).corr()
mean_corr = correlations.values[np.triu_indices_from(correlations.values, k=1)].mean()
print(f"  Average pairwise correlation: {mean_corr:.4f}")
print(
    f"  Interpretation: {'High' if mean_corr > 0.9 else 'Moderate' if mean_corr > 0.7 else 'Good'} diversity"
)

# Recommendations
print(f"\n💡 Recommendations:")
print(
    f"  1. {'Use ensemble methods for improved accuracy' if improvement > 0 else 'Individual models perform well'}"
)
print(f"  2. Focus on models in confidence set: {confidence_set}")
print(f"  3. Consider {results_df.iloc[0]['Model']} for production use")
print(f"  4. Monitor model performance over time and update weights dynamically")
print(
    f"  5. {'Increase model diversity for better ensemble performance' if mean_corr > 0.9 else 'Good model diversity achieved'}"
)

print("\n" + "=" * 60)
print("✅ Analysis Complete!")
print("=" * 60)